# importing libraries and dataset

In [13]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import accuracy_score

In [14]:
dataset_path = r'C:\Users\kunal\OneDrive\Desktop\Projects\coded\facemask-detector\Dataset'
image_size = (150,150)
batch_size = 30

In [15]:
train_dataset = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.30,
    subset='training',
    seed = 123,
    image_size = image_size,
    batch_size = batch_size
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split = 0.3,
    subset = 'validation',
    seed = 123,
    image_size=image_size,
    batch_size=batch_size
)

val_batches = tf.data.experimental.cardinality(validation_dataset)

# Move half of the validation batches into a new test_dataset
test_dataset = validation_dataset.take(val_batches // 2)

# Keep the other half as the validation_dataset
validation_dataset = validation_dataset.skip(val_batches // 2)




Found 5988 files belonging to 2 classes.
Using 4192 files for training.
Found 5988 files belonging to 2 classes.
Using 1796 files for validation.


# Building the model


In [16]:
from tensorflow.keras.applications import MobileNetV2

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

# 1. Load the pre-trained MobileNetV2 base model
base_model = MobileNetV2(input_shape=(150, 150, 3), include_top=False, weights='imagenet')

# 2. Freeze the base model to prevent destroying its pre-trained intelligence
base_model.trainable = False

# 3. Build our final classification model on top of it
model = keras.Sequential([
    data_augmentation,
    # MobileNetV2 requires pixels to be scaled from -1 to 1 instead of 0 to 1
    layers.Rescaling(1./127.5, offset=-1),
    
    base_model,
    
    # Compress the 2D spatial features into a 1D vector (better than Flatten for Transfer Learning)
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    
    layers.Dense(1, activation='sigmoid'),
])


C:\Users\kunal\AppData\Local\Temp\ipykernel_11640\298864163.py:10: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(input_shape=(150, 150, 3), include_top=False, weights='imagenet')


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [17]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [18]:
model.fit(train_dataset, validation_data=(validation_dataset), epochs=10)

Epoch 1/10


140/140 ━━━━━━━━━━━━━━━━━━━━ 82s 512ms/step - accuracy: 0.9349 - loss: 0.1638 - val_accuracy: 0.9833 - val_loss: 0.0442
Epoch 2/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 69s 490ms/step - accuracy: 0.9819 - loss: 0.0535 - val_accuracy: 0.9900 - val_loss: 0.0280
Epoch 3/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 73s 521ms/step - accuracy: 0.9847 - loss: 0.0433 - val_accuracy: 0.9888 - val_loss: 0.0266
Epoch 4/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 73s 525ms/step - accuracy: 0.9874 - loss: 0.0354 - val_accuracy: 0.9911 - val_loss: 0.0237
Epoch 5/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 80s 572ms/step - accuracy: 0.9874 - loss: 0.0329 - val_accuracy: 0.9944 - val_loss: 0.0163
Epoch 6/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 74s 531ms/step - accuracy: 0.9900 - loss: 0.0262 - val_accuracy: 0.9933 - val_loss: 0.0209
Epoch 7/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 70s 503ms/step - accuracy: 0.9881 - loss: 0.0308 - val_accuracy: 0.9933 - val_loss: 0.0132
Epoch 8/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 81s 583ms/step - accuracy: 0.9909 - loss: 0.0246 - val

# model metrics

In [19]:
test_loss, test_accuracy = model.evaluate(test_dataset)

print(f"Test Accuracy: {test_accuracy:.2%}")
print(f"Test Loss: {test_loss:.4f}")

30/30 ━━━━━━━━━━━━━━━━━━━━ 12s 377ms/step - accuracy: 0.9978 - loss: 0.0155
Test Accuracy: 99.78%
Test Loss: 0.0155


# saving model

In [20]:
print('saving model as model.keras')
model.save('model.keras')
print('saved!')


saving model as model.keras
saved!
